# Quantum Volume and CLOPS at pulse level

This notebook runs the Quantum Volume (QV) and CLOPS benchmarks at pulse level. The circuits are generated and
transpiled as in the circuit-level benchmarks, but they are then compiled to pulses on the client and submitted
directly to the quantum computer. This gives control over the compiler settings, for example the reset wait
duration or the dynamical decoupling sequences, which are not available at circuit level.

Requirements:

* A real backend: pulse-level execution needs the backend's `IQMClient`, so simulators and fake backends are not supported.
* A fixed physical layout: the circuits are compiled for the configured qubits, so `physical_layout` must be `"fixed"` (the default).

## Connect to the backend

In [ ]:
import os
from iqm.qiskit_iqm import IQMProvider

token = ""
os.environ["IQM_TOKEN"] = token
quantum_computer = "" # provide actual quantum computer name. i.e. "emerald", "garnet", "sirius"
iqm_server_url = "https://resonance.iqm.tech/" # provide actual IQM server URL
os.environ["IQM_SERVER_URL"] = iqm_server_url

provider = IQMProvider(iqm_server_url, quantum_computer=quantum_computer)
backend = provider.get_backend()

## Choose the qubit layout

Both benchmarks run on a single layout of physical qubit indices. CLOPS should be measured on the same layout
(and with the same transpilation options) that was used to establish QV.

In [ ]:
layout = [32, 33, 40, 41, 24, 25, 31]  # replace with the layout to benchmark

## Pulse settings

Compiler settings are passed through the `pulse_settings` field of the configurations, as a dictionary from a dotted
path in the compiler's settings tree to the value to set. The full settings tree can be inspected from the backend's
compiler:

In [ ]:
compiler = backend.client.get_standard_compiler()
settings = compiler.get_settings([])

The benchmarks build the settings tree for the benchmarked qubits only. Paths for qubits or couplers outside the
layout are skipped, so a dictionary built for the whole chip, as below, can be used for any layout. The log reports how
many paths were applied.

### Reset

By default the qubits are reset passively, by waiting for them to relax. The wait duration of every qubit and
coupler can be set as follows:

In [ ]:
reset_wait = {
    f"gates.reset_wait.reset_wait.{locus}.duration": 200e-6
    for locus, _ in settings.gates.reset_wait.reset_wait.child_nodes
}

Alternatively, the qubits can be reset actively with the `active_reset_cycles` field of the configurations. With
active reset the reset wait duration is not used.

### Dynamical decoupling

The `use_dd` and `dd_strategy` fields of the configurations are not used at pulse level; dynamical decoupling is
enabled through `pulse_settings` instead. Below, a minimal strategy (two extra single-qubit gates per idle) is applied
to the qubits with a single-qubit gate fidelity above 99.8%.

In [ ]:
from iqm.benchmarks.utils import extract_fidelities_unified
from iqm.iqm_client.models import DDStrategy

_, calibration_data = extract_fidelities_unified(backend=backend)
dd_qubits = [
    f"QB{qubit + 1}"
    for qubit, fidelity in calibration_data["fidelity_1qb_gates_averaged"].items()
    if fidelity > 0.998
]
strategy = DDStrategy(gate_sequences=[(2, "XX", "center")], target_qubits=dd_qubits)

dd_path = "stages.dynamical_decoupling.apply_dd_strategy"
ratios, patterns, aligns = zip(*strategy.gate_sequences)
dd_settings = {
    f"{dd_path}.dd_is_disabled": False,
    f"{dd_path}.use_standard_dd_strategy": False,
    f"{dd_path}.DDStrategy_gate_sequences_ratio": list(ratios),
    f"{dd_path}.DDStrategy_gate_sequences_gate_pattern_xy": list(patterns),
    f"{dd_path}.DDStrategy_gate_sequences_align": list(aligns),
    f"{dd_path}.DDStrategy_merge_contiguous_waits": strategy.merge_contiguous_waits,
    f"{dd_path}.DDStrategy_skip_leading_wait": strategy.skip_leading_wait,
    f"{dd_path}.DDStrategy_skip_trailing_wait": strategy.skip_trailing_wait,
    f"{dd_path}.DDStrategy_target_qubits": sorted(strategy.target_qubits),
}

To use the compiler's standard sequences instead, set only `dd_is_disabled` to `False` and
`use_standard_dd_strategy` to `True`.

## Quantum Volume

The configuration extends the circuit-level `QuantumVolumeConfiguration` with the pulse-level fields:

* `pulse_settings`: the settings described above.
* `seed`: makes the QV circuits reproducible. With the same seed, the circuit-level and pulse-level benchmarks generate the same circuits.
* `num_transpile_seeds`: transpiles each circuit with this many seeds and keeps the one with the fewest CZ gates. Increasing seed count improves performance at the cost of longer transpilation time. 
* `bootstrap_analysis`: also reports the bootstrap confidence bound of Baldwin et al., Quantum 6, 707 (2022), next to the default one.
* `rem`: readout error mitigation (M3). The readout calibration circuits are compiled and executed at pulse level
  with the same `pulse_settings` and `active_reset_cycles` as the QV circuits. This matters because the
  calibration measures everything between the reset and the readout: a shortened `reset_wait` leaves more
  residual excited population, and a calibration with the default reset wait would not capture it, so the
  mitigation would not match the executed experiment. The dynamical decoupling settings have no effect on the
  calibration circuits, which have no idle periods. Note that with matching calibration, REM also removes the
  state-preparation error added by a short `reset_wait`: to compare reset wait durations, use the raw heavy
  output probability, and read the mitigated one as the readout-corrected value.

In [ ]:
from iqm.benchmarks.quantum_volume.quantum_volume_pulse import (
    QuantumVolumePulseBenchmark,
    QuantumVolumePulseConfiguration,
)

EXAMPLE_QV = QuantumVolumePulseConfiguration(
    num_circuits=500,
    shots=2**8,
    num_sigmas=2,
    choose_qubits_routine="custom",
    custom_qubits_array=[layout],
    rem=True,
    mit_shots=1_000,
    pulse_settings={**reset_wait, **dd_settings},
    seed=0,
    num_transpile_seeds=10,
    bootstrap_analysis=True,
)

### Run the experiment

In [ ]:
benchmark_qv = QuantumVolumePulseBenchmark(backend, EXAMPLE_QV)
run_qv = benchmark_qv.run()

### Analyze the results

The observations contain, per layout, the average heavy output probability, whether the test passed and the
resulting QV, for the raw data and, with `rem=True`, for the readout-mitigated data. With `bootstrap_analysis=True`
they also contain the bootstrap lower bound and the corresponding pass/fail result.

In [ ]:
result_qv = benchmark_qv.analyze()
for observation in result_qv.observations:
    if observation.uncertainty is not None:
        print(f"{observation.name}: {observation.value} +/- {observation.uncertainty}")
    else:
        print(f"{observation.name}: {observation.value}")

In [ ]:
result_qv.plot_all()

## CLOPS

CLOPS measures how fast parametrized QV circuits are executed. Following the definition in arXiv:2110.14108, 100
parametrized circuit templates are generated and transpiled before the timer starts, together with the pulse
compiler. Then, for each of the 10 parameter updates, the timed loop assigns new parameters, compiles the circuits to
pulses, submits them and retrieves the results. The parameters of every update after the first are seeded with the
counts of the previous update, so an update cannot start before the previous one has returned.

Besides `pulse_settings`, the configuration has:

* `seed`: seeds the circuit templates and the parameters of the first update.
* `polling_interval`: seconds between job status checks. It is part of the timed loop, so it is kept short.

`num_circuits`, `num_updates` and `num_shots` are 100, 10 and 100 by definition.

In [ ]:
from iqm.benchmarks.quantum_volume.clops_pulse import CLOPSPulseBenchmark, CLOPSPulseConfiguration

EXAMPLE_CLOPS = CLOPSPulseConfiguration(
    qubits=layout,
    num_circuits=100,
    num_updates=10,
    num_shots=100,
    pulse_settings={**reset_wait},
    seed=0,
)

### Run the experiment

In [ ]:
benchmark_clops = CLOPSPulseBenchmark(backend, EXAMPLE_CLOPS)
run_clops = benchmark_clops.run()

### Analyze the results

The observation is CLOPS_v. The plots show the breakdown of the CLOPS time into the remote job times and the
client-side times (parameter assignment, compilation, submission and retrieval), and one plot per update with the
timeline of the job on the server.

In [ ]:
result_clops = benchmark_clops.analyze()
result_clops.observations

In [ ]:
result_clops.plot_all()

## Comparison with circuit-level CLOPS

With the same `seed` and layout, the circuit-level benchmark uses the same circuit templates and first-update
parameters, so the two CLOPS values can be compared directly.

In [ ]:
from iqm.benchmarks.quantum_volume.clops import CLOPSBenchmark, CLOPSConfiguration

EXAMPLE_CLOPS_CIRCUIT = CLOPSConfiguration(
    qubits=layout,
    num_circuits=100,
    num_updates=10,
    num_shots=100,
    seed=0,
)
benchmark_clops_circuit = CLOPSBenchmark(backend, EXAMPLE_CLOPS_CIRCUIT)
run_clops_circuit = benchmark_clops_circuit.run()
result_clops_circuit = benchmark_clops_circuit.analyze()
result_clops_circuit.observations